In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps as UTC datetimes
df_metric = pd.read_csv("metric.csv")
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

df_log = pd.read_csv("log.csv")
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

df_error = pd.read_csv("error_logs.csv")
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Helper: compact file summary (min/max timestamps and row count)
def file_summary(df):
    if df.shape[0] == 0:
        return pd.DataFrame([{"min_timestamp": pd.NaT, "max_timestamp": pd.NaT, "n_rows": 0}])
    return pd.DataFrame([{
        "min_timestamp": df['timestamp'].min(),
        "max_timestamp": df['timestamp'].max(),
        "n_rows": df.shape[0]
    }])

# Metric summary
metric_summary = file_summary(df_metric)
metric_cmdb_counts = df_metric['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
metric_kpi_counts = df_metric['kpi_name'].value_counts().head(20).rename_axis('kpi_name').reset_index(name='count')

# Trace summary
trace_summary = file_summary(df_trace)
trace_cmdb_counts = df_trace['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
trace_name_counts = df_trace['trace_name'].value_counts().head(20).rename_axis('trace_name').reset_index(name='count')

# Log summary
log_summary = file_summary(df_log)
log_cmdb_counts = df_log['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
log_name_counts = df_log['log_name'].value_counts().head(20).rename_axis('log_name').reset_index(name='count')

# Error logs summary and samples (up to 10)
error_summary = file_summary(df_error)
error_cmdb_counts = df_error['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
error_samples = df_error.sort_values('timestamp')[['timestamp', 'cmdb_id', 'message']].head(10)

# Display compact summaries (as variables; IPython will render them)
metric_summary, metric_cmdb_counts, metric_kpi_counts, \
trace_summary, trace_cmdb_counts, trace_name_counts, \
log_summary, log_cmdb_counts, log_name_counts, \
error_summary, error_cmdb_counts, error_samples

```
Out[1]:
```
Summary (plain English):

Metric data (metric.csv)
- Time range: 2024-01-19 22:52:00 UTC — 2024-01-19 23:16:00 UTC
- Row count: 1850
- Top cmdb_id (counts, top entries): catalogue (175), carts (150), front-end (150), orders (150), user (150), shipping (150), payment (150), orders-db (100), catalogue-db (100), carts-db (100), queue-master (100), session-db (100), rabbitmq (100), user-db (100), rabbitmq-exporter (75)
- Top kpi_name (counts): cpu (375), socket (375), mem (375), diskio (200), latency-50 (175), latency-90 (175), workload (175)

Trace data (trace.csv)
- File appears empty: no rows (n_rows = 0)
- No time range, no cmdb_id or trace_name entries available

Log data (log.csv)
- Time range: 2024-01-19 22:52:00 UTC — 2024-01-19 23:16:00 UTC
- Row count: 456
- Top cmdb_id (counts): carts-db (50), catalogue (50), front-end (50), shipping (50), orders (50), payment (50), queue-master (50), user (50), carts (44), session-db (10), user-db (2)
- Top log_name (counts): log.error_count (228), log.row_count (228)

Error logs (error_logs.csv)
- File appears empty: no rows (n_rows = 0)
- No time range, no cmdb_id counts, no example messages

Key takeaway
- Metric and log telemetry are present for the 22:52–23:16 UTC window; metrics show many services and CPU/socket/memory KPIs are most common. Trace and error_log files contain no data for this dataset.

The original code execution output of IPython Kernel is also provided below for reference:

(              min_timestamp             max_timestamp  n_rows
0 2024-01-19 22:52:00+00:00 2024-01-19 23:16:00+00:00    1850,               cmdb_id  count
0           catalogue    175
1               carts    150
2           front-end    150
3              orders    150
4                user    150
5            shipping    150
6             payment    150
7           orders-db    100
8        catalogue-db    100
9            carts-db    100
10       queue-master    100
11         session-db    100
12           rabbitmq    100
13            user-db    100
14  rabbitmq-exporter     75,      kpi_name  count
0         cpu    375
1      socket    375
2         mem    375
3      diskio    200
4  latency-50    175
5  latency-90    175
6    workload    175,   min_timestamp max_timestamp  n_rows
0           NaT           NaT       0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [trace_name, count]
Index: [],               min_timestamp             max_timestamp  n_rows
0 2024-01-19 22:52:00+00:00 2024-01-19 23:16:00+00:00     456,          cmdb_id  count
0       carts-db     50
1      catalogue     50
2      front-end     50
3       shipping     50
4         orders     50
5        payment     50
6   queue-master     50
7           user     50
8          carts     44
9     session-db     10
10       user-db      2,           log_name  count
0  log.error_count    228
1    log.row_count    228,   min_timestamp max_timestamp  n_rows
0           NaT           NaT       0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse df_metric if available, otherwise load and parse timestamps
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp is parsed and tz-aware
    if df_metric['timestamp'].dtype != 'datetime64[ns, UTC]':
        df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# 1) Compute global percentiles P95, P90, P5 and total counts per (cmdb_id, kpi_name)
group = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
full_counts = group.count().reset_index(name='n_total_points_in_full_series')

P95 = group.quantile(0.95).reset_index(name='P95')
P90 = group.quantile(0.90).reset_index(name='P90')
P5  = group.quantile(0.05).reset_index(name='P5')

# Merge thresholds and counts into one dataframe (one row per pair)
df_pairs = full_counts.merge(P95, on=['cmdb_id','kpi_name']).merge(P90, on=['cmdb_id','kpi_name']).merge(P5, on=['cmdb_id','kpi_name'])

# 2) Filter metric rows to incident window (UTC)
start = pd.Timestamp('2024-01-19 22:49:37', tz='UTC')
end   = pd.Timestamp('2024-01-19 23:19:37', tz='UTC')
df_window = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)].copy()

# 3) Attach thresholds to window rows for comparison
df_window = df_window.merge(df_pairs[['cmdb_id','kpi_name','P95','P90','P5']], on=['cmdb_id','kpi_name'], how='left')

# If any pair in the window had no full-series thresholds (shouldn't happen), drop those rows
df_window = df_window.dropna(subset=['P95','P90','P5'])

# 4) Compute booleans for breaches
df_window['ge_P95'] = df_window['value'] >= df_window['P95']
df_window['ge_P90'] = df_window['value'] >= df_window['P90']
df_window['le_P5']  = df_window['value'] <= df_window['P5']

# 5) Aggregate per (cmdb_id, kpi_name) within the window
def summarize_window(g):
    n_window = int(len(g))
    n_ge95 = int(g['ge_P95'].sum())
    earliest_ge95 = g.loc[g['ge_P95'], 'timestamp'].min() if n_ge95 > 0 else pd.NaT
    n_ge90 = int(g['ge_P90'].sum())
    n_le5 = int(g['le_P5'].sum())
    earliest_le5 = g.loc[g['le_P5'], 'timestamp'].min() if n_le5 > 0 else pd.NaT
    return pd.Series({
        'n_points_in_window': n_window,
        'n_points_in_window_ge_P95': n_ge95,
        'earliest_timestamp_in_window_ge_P95': earliest_ge95,
        'n_points_in_window_ge_P90': n_ge90,
        'n_points_in_window_le_P5': n_le5,
        'earliest_timestamp_in_window_le_P5': earliest_le5
    })

window_summary = df_window.groupby(['cmdb_id','kpi_name']).apply(summarize_window).reset_index()

# 6) Combine full-series pairs with window summary (include pairs with zero points in window)
summary = df_pairs.merge(window_summary, on=['cmdb_id','kpi_name'], how='left')

# Fill NA counts with 0 and keep NaT for earliest timestamps where appropriate
count_cols = ['n_points_in_window','n_points_in_window_ge_P95','n_points_in_window_ge_P90','n_points_in_window_le_P5']
summary[count_cols] = summary[count_cols].fillna(0).astype(int)
# earliest timestamps: keep as-is (NaT where missing)

# 7) Select and order columns as requested
summary = summary[[
    'cmdb_id','kpi_name','n_total_points_in_full_series','n_points_in_window',
    'P95','P90','P5',
    'n_points_in_window_ge_P95','earliest_timestamp_in_window_ge_P95',
    'n_points_in_window_ge_P90','n_points_in_window_le_P5','earliest_timestamp_in_window_le_P5'
]]

# 8) Order by n_points_in_window_ge_P95 descending and limit to top 20
summary_top20 = summary.sort_values(by='n_points_in_window_ge_P95', ascending=False).head(20)

# Display the compact summary (top 20 rows)
summary_top20

```
Out[2]:
```
summary = (
    "Summary of metric.csv analysis (incident window 2024-01-19 22:49:37 UTC through 2024-01-19 23:19:37 UTC):\n\n"
    "Key finding: Several component-KPI pairs had sustained values at or above their global P95 for the entire window (25/25 points):\n"
    "- carts-db / socket: 25/25 points ≥ P95 (earliest breach 2024-01-19 22:52:00 UTC)\n"
    "- rabbitmq-exporter / socket: 25/25 ≥ P95 (earliest 2024-01-19 22:52:00 UTC)\n"
    "- rabbitmq / diskio: 25/25 ≥ P95 (earliest 2024-01-19 22:52:00 UTC)\n"
    "- queue-master / socket: 25/25 ≥ P95 (earliest 2024-01-19 22:52:00 UTC)\n"
    "- rabbitmq-exporter / mem: 25/25 ≥ P95 (earliest 2024-01-19 22:52:00 UTC)\n"
    "- rabbitmq / socket: 25/25 ≥ P95 (earliest 2024-01-19 22:52:00 UTC)\n"
    "- session-db / socket: 25/25 ≥ P95 (earliest 2024-01-19 22:52:00 UTC)\n"
    "- orders-db / socket: 25/25 ≥ P95 (earliest 2024-01-19 22:52:00 UTC)\n"
    "- catalogue-db / diskio: 25/25 ≥ P95 (earliest 2024-01-19 22:52:00 UTC)\n"
    "- catalogue-db / socket: 25/25 ≥ P95 (earliest 2024-01-19 22:52:00 UTC)\n\n"
    "Other notable pairs (fewer but significant P95 breaches):\n"
    "- user-db / socket: 18/25 points ≥ P95 (earliest 2024-01-19 22:59:00 UTC)\n"
    "- catalogue / socket: 12/25 ≥ P95 (earliest 2024-01-19 23:05:00 UTC)\n"
    "- user / socket: 5/25 ≥ P95 (earliest 2024-01-19 23:12:00 UTC)\n"
    "- carts-db / mem, carts / mem, carts / socket, carts / latency-90, carts / latency-50, carts / cpu, carts-db / diskio: mostly 2–3 breaches each with earliest breaches between 22:52 and 23:15 UTC\n\n"
    "Additional context from the computed table:\n"
    "- Most component-KPI pairs had 25 total points in the window (full-series counts are also 25 for these pairs).\n"
    "- For many of the top offenders the earliest breach timestamp is 2024-01-19 22:52:00 UTC (i.e., immediately after the window started), indicating sustained high values through the window.\n\n"
    "Conclusion: Multiple services—particularly RabbitMQ-related components (rabbitmq, rabbitmq-exporter), queue-master, various DB sockets (orders-db, session-db, catalogue-db, carts-db), and catalogue/socket—showed persistent KPI values at or above their historical P95 during the incident window. These are the strongest signals of abnormal behavior in metric.csv for the given window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  n_total_points_in_full_series  n_points_in_window           P95           P90            P5  n_points_in_window_ge_P95 earliest_timestamp_in_window_ge_P95  n_points_in_window_ge_P90  n_points_in_window_le_P5 earliest_timestamp_in_window_le_P5
9            carts-db      socket                             25                  25  6.000000e+00  6.000000e+00  6.000000e+00                         25           2024-01-19 22:52:00+00:00                         25                        24          2024-01-19 22:52:00+00:00
53  rabbitmq-exporter      socket                             25                  25  1.000000e+00  1.000000e+00  1.000000e+00                         25           2024-01-19 22:52:00+00:00                         25                        25          2024-01-19 22:52:00+00:00
48           rabbitmq      diskio                             25                  25  0.000000e+00  0.000000e+00  0.000000e+00                         25           2024-01-19 22:52:00+00:00                         25                        25          2024-01-19 22:52:00+00:00
46       queue-master      socket                             25                  25  3.000000e+00  3.000000e+00  3.000000e+00                         25           2024-01-19 22:52:00+00:00                         25                        25          2024-01-19 22:52:00+00:00
52  rabbitmq-exporter         mem                             25                  25  4.669440e+06  4.669440e+06  4.669440e+06                         25           2024-01-19 22:52:00+00:00                         25                        25          2024-01-19 22:52:00+00:00
50           rabbitmq      socket                             25                  25  1.100000e+01  1.100000e+01  1.100000e+01                         25           2024-01-19 22:52:00+00:00                         25                        25          2024-01-19 22:52:00+00:00
57         session-db      socket                             25                  25  3.000000e+00  3.000000e+00  3.000000e+00                         25           2024-01-19 22:52:00+00:00                         25                        25          2024-01-19 22:52:00+00:00
36          orders-db      socket                             25                  25  6.000000e+00  6.000000e+00  6.000000e+00                         25           2024-01-19 22:52:00+00:00                         25                        25          2024-01-19 22:52:00+00:00
18       catalogue-db      diskio                             25                  25  0.000000e+00  0.000000e+00  0.000000e+00                         25           2024-01-19 22:52:00+00:00                         25                        25          2024-01-19 22:52:00+00:00
20       catalogue-db      socket                             25                  25  4.000000e+00  4.000000e+00  4.000000e+00                         25           2024-01-19 22:52:00+00:00                         25                        25          2024-01-19 22:52:00+00:00
73            user-db      socket                             25                  25  8.000000e+00  8.000000e+00  7.000000e+00                         18           2024-01-19 22:59:00+00:00                         18                         6          2024-01-19 22:52:00+00:00
15          catalogue      socket                             25                  25  8.000000e+00  8.000000e+00  5.000000e+00                         12           2024-01-19 23:05:00+00:00                         12                        12          2024-01-19 22:52:00+00:00
68               user      socket                             25                  25  1.700000e+01  1.700000e+01  1.500000e+01                          5           2024-01-19 23:12:00+00:00                          5                         6          2024-01-19 22:52:00+00:00
8            carts-db         mem                             25                  25  8.052453e+07  8.029229e+07  7.412270e+07                          2           2024-01-19 23:15:00+00:00                          3                         2          2024-01-19 22:52:00+00:00
3               carts         mem                             25                  25  2.097919e+08  2.096834e+08  2.057959e+08                          2           2024-01-19 23:11:00+00:00                          3                         2          2024-01-19 22:52:00+00:00
4               carts      socket                             25                  25  1.091333e+01  1.076000e+01  9.760000e+00                          2           2024-01-19 22:57:00+00:00                          3                         2          2024-01-19 22:53:00+00:00
2               carts  latency-90                             25                  25  2.861734e-02  2.726000e-02  2.245729e-02                          2           2024-01-19 23:05:00+00:00                          3                         2          2024-01-19 22:53:00+00:00
1               carts  latency-50                             25                  25  1.608867e-02  1.595698e-02  1.164941e-02                          2           2024-01-19 23:07:00+00:00                          3                         2          2024-01-19 22:53:00+00:00
0               carts         cpu                             25                  25  2.350967e+00  2.127047e+00  1.363188e+00                          2           2024-01-19 22:54:00+00:00                          3                         2          2024-01-19 23:15:00+00:00
7            carts-db      diskio                             25                  25  2.123998e+06  2.058979e+06  1.680320e+06                          2           2024-01-19 23:04:00+00:00                          3                         2          2024-01-19 22:56:00+00:00```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse df_metric if available; otherwise load and parse timestamps
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp is parsed and UTC-aware
    if not pd.api.types.is_datetime64_any_dtype(df_metric['timestamp']):
        df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
    else:
        # ensure tz-aware (some environments may have tz-naive)
        if df_metric['timestamp'].dt.tz is None:
            df_metric['timestamp'] = df_metric['timestamp'].dt.tz_localize('UTC')

# Service-level components to analyze
services = ['carts', 'catalogue', 'orders', 'payment', 'user']

# Filter full-series to only those services for computing global thresholds
df_services_full = df_metric[df_metric['cmdb_id'].isin(services)].copy()

# 1) Compute global percentiles and full-series counts per (cmdb_id, kpi_name)
group = df_services_full.groupby(['cmdb_id', 'kpi_name'])['value']
pairs_counts = group.count().reset_index(name='n_total_points_in_full_series')
P95 = group.quantile(0.95).reset_index(name='P95')
P90 = group.quantile(0.90).reset_index(name='P90')
P5  = group.quantile(0.05).reset_index(name='P5')

df_pairs = pairs_counts.merge(P95, on=['cmdb_id','kpi_name']).merge(P90, on=['cmdb_id','kpi_name']).merge(P5, on=['cmdb_id','kpi_name'])

# 2) Filter rows to the incident window (UTC)
start = pd.Timestamp('2024-01-19 22:49:37', tz='UTC')
end   = pd.Timestamp('2024-01-19 23:19:37', tz='UTC')
df_window = df_metric[
    (df_metric['cmdb_id'].isin(services)) &
    (df_metric['timestamp'] >= start) &
    (df_metric['timestamp'] <= end)
].copy()

# 3) Attach thresholds to window rows for comparison (join on cmdb_id & kpi_name)
df_window = df_window.merge(df_pairs[['cmdb_id','kpi_name','P95','P90','P5']], on=['cmdb_id','kpi_name'], how='left')

# Drop any rows in window where we couldn't compute thresholds (should be rare)
df_window = df_window.dropna(subset=['P95','P90','P5'])

# 4) Compute breach booleans
df_window['ge_P95'] = df_window['value'] >= df_window['P95']
df_window['ge_P90'] = df_window['value'] >= df_window['P90']
df_window['le_P5']  = df_window['value'] <= df_window['P5']

# 5) Aggregate per (cmdb_id, kpi_name) within the window
def summarize_window(g):
    n_window = int(len(g))
    n_ge95 = int(g['ge_P95'].sum())
    earliest_ge95 = g.loc[g['ge_P95'], 'timestamp'].min() if n_ge95 > 0 else pd.NaT
    n_ge90 = int(g['ge_P90'].sum())
    n_le5 = int(g['le_P5'].sum())
    earliest_le5 = g.loc[g['le_P5'], 'timestamp'].min() if n_le5 > 0 else pd.NaT
    return pd.Series({
        'n_points_in_window': n_window,
        'n_points_in_window_ge_P95': n_ge95,
        'earliest_timestamp_in_window_ge_P95': earliest_ge95,
        'n_points_in_window_ge_P90': n_ge90,
        'n_points_in_window_le_P5': n_le5,
        'earliest_timestamp_in_window_le_P5': earliest_le5
    })

window_summary = df_window.groupby(['cmdb_id','kpi_name']).apply(summarize_window).reset_index()

# 6) Combine full-series pairs with window summary (include pairs with zero points in window)
summary = df_pairs.merge(window_summary, on=['cmdb_id','kpi_name'], how='left')

# Fill NA counts with 0 (for pairs with no points in window); keep NaT for earliest timestamps
count_cols = ['n_points_in_window','n_points_in_window_ge_P95','n_points_in_window_ge_P90','n_points_in_window_le_P5']
summary[count_cols] = summary[count_cols].fillna(0).astype(int)

# 7) Select and order columns as requested
summary = summary[[
    'cmdb_id','kpi_name','n_total_points_in_full_series','n_points_in_window',
    'P95','P90','P5',
    'n_points_in_window_ge_P95','earliest_timestamp_in_window_ge_P95',
    'n_points_in_window_ge_P90','n_points_in_window_le_P5','earliest_timestamp_in_window_le_P5'
]]

# 8) Order by n_points_in_window_ge_P95 descending and limit to top 20
summary_top20 = summary.sort_values(by='n_points_in_window_ge_P95', ascending=False).head(20)

# Display the compact summary (top 20 rows)
summary_top20

```
Out[3]:
```
Summary (service-level metrics: carts, catalogue, orders, payment, user) for incident window 2024-01-19 22:49:37 → 2024-01-19 23:19:37 UTC:

High-level
- Every reported (cmdb_id, kpi_name) pair has 25 total points in the full series and 25 points in the incident window (i.e., the window contains the entire available series for these pairs).
- We computed global P95, P90 and P5 per pair from the full series, then counted breaches inside the window.

Top findings (ordered by number of P95 breaches)
1. catalogue / socket
   - n_total_points: 25, n_points_in_window: 25
   - P95 = 8.0, P90 = 8.0, P5 = 5.0
   - n_points_in_window_ge_P95 = 12 (earliest ≥P95 at 2024-01-19 23:05:00 UTC)
   - n_points_in_window_ge_P90 = 12
   - n_points_in_window_le_P5 = 12 (earliest ≤P5 at 2024-01-19 22:52:00 UTC)

2. user / socket
   - n_total_points: 25, n_points_in_window: 25
   - P95 = 17.0, P90 = 17.0, P5 = 15.0
   - n_points_in_window_ge_P95 = 5 (earliest ≥P95 at 2024-01-19 23:12:00 UTC)
   - n_points_in_window_ge_P90 = 5
   - n_points_in_window_le_P5 = 6 (earliest ≤P5 at 2024-01-19 22:52:00 UTC)

Several other service‑KPIs show 1–3 P95 breaches (all listed below had 2 breaches unless noted):
- carts / latency-50 — P95 ≈ 0.0160887, n_ge_P95 = 2 (earliest ≥P95 23:07:00 UTC)
- carts / mem — very large mem values, n_ge_P95 = 2 (earliest 23:11:00 UTC)
- carts / socket — P95 ≈ 10.91333, n_ge_P95 = 2 (earliest 22:57:00 UTC)
- carts / workload — n_ge_P95 = 2 (earliest 22:58:00 UTC)
- carts / cpu — P95 ≈ 2.350967, n_ge_P95 = 2 (earliest 22:54:00 UTC)
- catalogue / cpu — n_ge_P95 = 2 (earliest 23:10:00 UTC)
- catalogue / diskio — n_ge_P95 = 2 (earliest 23:10:00 UTC); note n_points_in_window_le_P5 = 10 (earliest ≤P5 at 22:54:00 UTC)
- catalogue / latency-90 and / latency-50 — each n_ge_P95 = 2 (earliest ~23:09–23:10 UTC)
- catalogue / mem and / workload — each n_ge_P95 = 2 (earliest 23:10 and 22:57 UTC respectively)
- orders / cpu, orders / latency-50, orders / latency-90, orders / socket, orders / mem — each n_ge_P95 = 2 (earliest timestamps between ~22:56 and 23:11 UTC)
- payment / cpu — n_ge_P95 = 2 (earliest 22:57:00 UTC)

Interpretation
- The strongest anomaly signal is catalogue/socket (12/25 points ≥ its P95) indicating sustained elevated socket metric values beginning ~23:05 UTC.
- user/socket also shows multiple P95 breaches (5/25) beginning ~23:12 UTC.
- Many carts, catalogue and orders KPIs show small numbers of P95 breaches (mostly 2), indicating occasional spikes during the window.
- catalogue/diskio also shows many low-end points (10 points ≤ P5) beginning ~22:54 UTC, indicating some values unusually low for that KPI during the window.

Bottom line
- The most prominent service-level abnormality in metric.csv during this window is with catalogue/socket (largest and sustained P95 breaches), followed by user/socket. Other service KPIs exhibit minor or intermittent breaches.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  n_total_points_in_full_series  n_points_in_window           P95           P90            P5  n_points_in_window_ge_P95 earliest_timestamp_in_window_ge_P95  n_points_in_window_ge_P90  n_points_in_window_le_P5 earliest_timestamp_in_window_le_P5
11  catalogue      socket                             25                  25  8.000000e+00  8.000000e+00  5.000000e+00                         12           2024-01-19 23:05:00+00:00                         12                        12          2024-01-19 22:52:00+00:00
29       user      socket                             25                  25  1.700000e+01  1.700000e+01  1.500000e+01                          5           2024-01-19 23:12:00+00:00                          5                         6          2024-01-19 22:52:00+00:00
1       carts  latency-50                             25                  25  1.608867e-02  1.595698e-02  1.164941e-02                          2           2024-01-19 23:07:00+00:00                          3                         2          2024-01-19 22:53:00+00:00
3       carts         mem                             25                  25  2.097919e+08  2.096834e+08  2.057959e+08                          2           2024-01-19 23:11:00+00:00                          3                         2          2024-01-19 22:52:00+00:00
4       carts      socket                             25                  25  1.091333e+01  1.076000e+01  9.760000e+00                          2           2024-01-19 22:57:00+00:00                          3                         2          2024-01-19 22:53:00+00:00
5       carts    workload                             25                  25  8.516340e+00  8.449647e+00  7.815337e+00                          2           2024-01-19 22:58:00+00:00                          3                         2          2024-01-19 23:05:00+00:00
0       carts         cpu                             25                  25  2.350967e+00  2.127047e+00  1.363188e+00                          2           2024-01-19 22:54:00+00:00                          3                         2          2024-01-19 23:15:00+00:00
6   catalogue         cpu                             25                  25  2.729340e+01  2.712250e+01  1.786273e-01                          2           2024-01-19 23:10:00+00:00                          3                         2          2024-01-19 23:00:00+00:00
7   catalogue      diskio                             25                  25  2.670087e+10  2.633973e+10  0.000000e+00                          2           2024-01-19 23:10:00+00:00                          3                        10          2024-01-19 22:54:00+00:00
9   catalogue  latency-90                             25                  25  4.887725e-03  4.850815e-03  4.663430e-03                          2           2024-01-19 23:09:00+00:00                          3                         2          2024-01-19 22:56:00+00:00
8   catalogue  latency-50                             25                  25  3.148791e-03  3.118559e-03  3.035239e-03                          2           2024-01-19 23:10:00+00:00                          3                         2          2024-01-19 22:56:00+00:00
10  catalogue         mem                             25                  25  6.291138e+08  6.290846e+08  6.290869e+06                          2           2024-01-19 23:10:00+00:00                          3                         2          2024-01-19 22:52:00+00:00
12  catalogue    workload                             25                  25  4.219656e+00  4.198476e+00  3.966193e+00                          2           2024-01-19 22:57:00+00:00                          3                         2          2024-01-19 23:06:00+00:00
13     orders         cpu                             25                  25  2.094577e+00  2.004078e+00  1.162408e+00                          2           2024-01-19 23:05:00+00:00                          3                         2          2024-01-19 23:08:00+00:00
2       carts  latency-90                             25                  25  2.861734e-02  2.726000e-02  2.245729e-02                          2           2024-01-19 23:05:00+00:00                          3                         2          2024-01-19 22:53:00+00:00
14     orders  latency-50                             25                  25  3.775879e-02  3.737217e-02  3.640904e-02                          2           2024-01-19 23:11:00+00:00                          3                         2          2024-01-19 22:53:00+00:00
15     orders  latency-90                             25                  25  4.819492e-02  4.777398e-02  4.728181e-02                          2           2024-01-19 23:11:00+00:00                          3                         2          2024-01-19 22:53:00+00:00
17     orders      socket                             25                  25  1.710667e+01  1.668667e+01  1.482667e+01                          2           2024-01-19 22:58:00+00:00                          3                         2          2024-01-19 23:03:00+00:00
16     orders         mem                             25                  25  3.198320e+08  3.198037e+08  3.191761e+08                          2           2024-01-19 22:56:00+00:00                          3                         2          2024-01-19 22:52:00+00:00
19    payment         cpu                             25                  25  9.746287e-02  9.635867e-02  7.966058e-02                          2           2024-01-19 22:57:00+00:00                          3                         2          2024-01-19 23:02:00+00:00```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse df_log if available; otherwise load and parse timestamps as UTC datetimes
if 'df_log' not in globals():
    df_log = pd.read_csv("log.csv")
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
else:
    if not pd.api.types.is_datetime64_any_dtype(df_log['timestamp']):
        df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
    else:
        if df_log['timestamp'].dt.tz is None:
            df_log['timestamp'] = df_log['timestamp'].dt.tz_localize('UTC')

# Incident window (UTC)
start = pd.Timestamp('2024-01-19 22:49:37', tz='UTC')
end   = pd.Timestamp('2024-01-19 23:19:37', tz='UTC')

# Filter rows to the incident window (inclusive)
df_window = df_log[(df_log['timestamp'] >= start) & (df_log['timestamp'] <= end)].copy()

# 1) Aggregate table grouped by cmdb_id and log_name
agg = df_window.groupby(['cmdb_id', 'log_name']).agg(
    n_rows_in_window = ('value', 'size'),
    sum_value = ('value', 'sum'),
    n_points_value_gt0 = ('value', lambda x: (x > 0).sum())
).reset_index()

# earliest_timestamp_value_gt0: compute from positive-value subset and merge
df_pos = df_window[df_window['value'] > 0]
earliest_pos = df_pos.groupby(['cmdb_id', 'log_name'])['timestamp'].min().reset_index(name='earliest_timestamp_value_gt0')

agg = agg.merge(earliest_pos, on=['cmdb_id', 'log_name'], how='left')

# Sort by sum_value descending and keep top 20
agg_top20 = agg.sort_values(by='sum_value', ascending=False).head(20)

# 2) Samples for candidate services: catalogue, user, carts (up to 20 rows each, sorted by timestamp)
candidates = ['catalogue', 'user', 'carts']
samples = {}
for svc in candidates:
    df_svc = df_window[df_window['cmdb_id'] == svc].sort_values('timestamp')[['timestamp','cmdb_id','log_name','value']].head(20)
    samples[svc] = df_svc

# Prepare outputs (compact). Display agg_top20 and up to 3 sample tables (total rows <= 60)
agg_top20, samples['catalogue'], samples['user'], samples['carts']

```
Out[4]:
```
Summary of log.csv within incident window 2024-01-19 22:49:37 → 2024-01-19 23:19:37 UTC

1) Aggregated counts (top rows by sum_value)
- front-end / log.row_count: n_rows_in_window=25, sum_value=47127, n_points_value_gt0=25, earliest_positive=2024-01-19 22:52:00 UTC
- user / log.row_count: n_rows_in_window=25, sum_value=14206, n_points_value_gt0=25, earliest_positive=2024-01-19 22:52:00 UTC
- queue-master / log.row_count: n_rows_in_window=25, sum_value=13239, n_points_value_gt0=25, earliest_positive=2024-01-19 22:52:00 UTC
- catalogue / log.row_count: n_rows_in_window=25, sum_value=3906, n_points_value_gt0=25, earliest_positive=2024-01-19 22:52:00 UTC
- orders / log.row_count: n_rows_in_window=25, sum_value=2942, n_points_value_gt0=25, earliest_positive=2024-01-19 22:52:00 UTC
- payment / log.row_count: n_rows_in_window=25, sum_value=2431, n_points_value_gt0=25, earliest_positive=2024-01-19 22:52:00 UTC
- shipping / log.row_count: n_rows_in_window=25, sum_value=1471, n_points_value_gt0=25, earliest_positive=2024-01-19 22:52:00 UTC
- carts / log.row_count: n_rows_in_window=22, sum_value=1451, n_points_value_gt0=22, earliest_positive=2024-01-19 22:52:00 UTC
- session-db / log.row_count: n_rows_in_window=5, sum_value=25, n_points_value_gt0=5, earliest_positive=2024-01-19 22:54:00 UTC
- carts-db / log.row_count: n_rows_in_window=25, sum_value=25, n_points_value_gt0=25, earliest_positive=2024-01-19 22:52:00 UTC
- user-db / log.row_count: n_rows_in_window=1, sum_value=1, n_points_value_gt0=1, earliest_positive=2024-01-19 22:58:00 UTC

Additional note on log.error_count:
- For all listed services (catalogue, carts, etc.) log.error_count rows exist in the window but sum_value = 0 and n_points_value_gt0 = 0 (no positive error counts); earliest_positive is NaT.

2) Samples for catalogue, user, carts (examples from the window, ordered by timestamp)
- Catalogue (up to 20 sample rows): repeated per-minute entries showing
  - log.row_count values around 160–168 per minute (e.g., 22:52 → 61 then 164 at 22:53, then ~161–168 each minute),
  - log.error_count = 0 for every sample,
  - earliest sample at 2024-01-19 22:52:00+00:00.

- User (up to 20 sample rows): repeated per-minute entries showing
  - log.row_count values high (~213 → 589 → 614 → 595 … up to ~617),
  - log.error_count = 0 for every sample,
  - earliest sample at 2024-01-19 22:52:00+00:00.

- Carts (up to 20 sample rows): per-minute entries showing
  - log.row_count values lower and more variable (examples: 75, 25, 75, 50, 100, 25, 75, 25, 125, 50),
  - log.error_count = 0 for every sample,
  - earliest sample at 2024-01-19 22:52:00+00:00.

Overall interpretation
- There are no positive log.error_count values for catalogue, user, or carts during the incident window (all error_count = 0).
- Log row counts show heavy traffic for front-end and user, moderate consistent traffic for catalogue, and variable lower traffic for carts.
- The earliest positive log activity for most services in the window starts at 2024-01-19 22:52:00 UTC.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id         log_name  n_rows_in_window  sum_value  n_points_value_gt0 earliest_timestamp_value_gt0
7      front-end    log.row_count                25      47127                  25    2024-01-19 22:52:00+00:00
19          user    log.row_count                25      14206                  25    2024-01-19 22:52:00+00:00
13  queue-master    log.row_count                25      13239                  25    2024-01-19 22:52:00+00:00
5      catalogue    log.row_count                25       3906                  25    2024-01-19 22:52:00+00:00
9         orders    log.row_count                25       2942                  25    2024-01-19 22:52:00+00:00
11       payment    log.row_count                25       2431                  25    2024-01-19 22:52:00+00:00
17      shipping    log.row_count                25       1471                  25    2024-01-19 22:52:00+00:00
1          carts    log.row_count                22       1451                  22    2024-01-19 22:52:00+00:00
15    session-db    log.row_count                 5         25                   5    2024-01-19 22:54:00+00:00
3       carts-db    log.row_count                25         25                  25    2024-01-19 22:52:00+00:00
21       user-db    log.row_count                 1          1                   1    2024-01-19 22:58:00+00:00
4      catalogue  log.error_count                25          0                   0                          NaT
0          carts  log.error_count                22          0                   0                          NaT
2       carts-db  log.error_count                25          0                   0                          NaT
12  queue-master  log.error_count                25          0                   0                          NaT
10       payment  log.error_count                25          0                   0                          NaT
6      front-end  log.error_count                25          0                   0                          NaT
8         orders  log.error_count                25          0                   0                          NaT
16      shipping  log.error_count                25          0                   0                          NaT
14    session-db  log.error_count                 5          0                   0                          NaT,                     timestamp    cmdb_id         log_name  value
4   2024-01-19 22:52:00+00:00  catalogue  log.error_count      0
5   2024-01-19 22:52:00+00:00  catalogue    log.row_count     61
22  2024-01-19 22:53:00+00:00  catalogue  log.error_count      0
23  2024-01-19 22:53:00+00:00  catalogue    log.row_count    164
38  2024-01-19 22:54:00+00:00  catalogue  log.error_count      0
39  2024-01-19 22:54:00+00:00  catalogue    log.row_count    161
58  2024-01-19 22:55:00+00:00  catalogue  log.error_count      0
59  2024-01-19 22:55:00+00:00  catalogue    log.row_count    168
76  2024-01-19 22:56:00+00:00  catalogue  log.error_count      0
77  2024-01-19 22:56:00+00:00  catalogue    log.row_count    163
94  2024-01-19 22:57:00+00:00  catalogue  log.error_count      0
95  2024-01-19 22:57:00+00:00  catalogue    log.row_count    165
111 2024-01-19 22:58:00+00:00  catalogue    log.row_count    168
110 2024-01-19 22:58:00+00:00  catalogue  log.error_count      0
130 2024-01-19 22:59:00+00:00  catalogue  log.error_count      0
131 2024-01-19 22:59:00+00:00  catalogue    log.row_count    161
150 2024-01-19 23:00:00+00:00  catalogue  log.error_count      0
151 2024-01-19 23:00:00+00:00  catalogue    log.row_count    167
168 2024-01-19 23:01:00+00:00  catalogue  log.error_count      0
169 2024-01-19 23:01:00+00:00  catalogue    log.row_count    160,                     timestamp cmdb_id         log_name  value
16  2024-01-19 22:52:00+00:00    user  log.error_count      0
17  2024-01-19 22:52:00+00:00    user    log.row_count    213
34  2024-01-19 22:53:00+00:00    user  log.error_count      0
35  2024-01-19 22:53:00+00:00    user    log.row_count    589
52  2024-01-19 22:54:00+00:00    user  log.error_count      0
53  2024-01-19 22:54:00+00:00    user    log.row_count    614
70  2024-01-19 22:55:00+00:00    user  log.error_count      0
71  2024-01-19 22:55:00+00:00    user    log.row_count    595
88  2024-01-19 22:56:00+00:00    user  log.error_count      0
89  2024-01-19 22:56:00+00:00    user    log.row_count    599
106 2024-01-19 22:57:00+00:00    user  log.error_count      0
107 2024-01-19 22:57:00+00:00    user    log.row_count    594
123 2024-01-19 22:58:00+00:00    user    log.row_count    617
122 2024-01-19 22:58:00+00:00    user  log.error_count      0
144 2024-01-19 22:59:00+00:00    user  log.error_count      0
145 2024-01-19 22:59:00+00:00    user    log.row_count    559
162 2024-01-19 23:00:00+00:00    user  log.error_count      0
163 2024-01-19 23:00:00+00:00    user    log.row_count    610
180 2024-01-19 23:01:00+00:00    user  log.error_count      0
181 2024-01-19 23:01:00+00:00    user    log.row_count    592,                     timestamp cmdb_id         log_name  value
0   2024-01-19 22:52:00+00:00   carts  log.error_count      0
1   2024-01-19 22:52:00+00:00   carts    log.row_count     75
18  2024-01-19 22:53:00+00:00   carts  log.error_count      0
19  2024-01-19 22:53:00+00:00   carts    log.row_count     25
54  2024-01-19 22:55:00+00:00   carts  log.error_count      0
55  2024-01-19 22:55:00+00:00   carts    log.row_count     75
72  2024-01-19 22:56:00+00:00   carts  log.error_count      0
73  2024-01-19 22:56:00+00:00   carts    log.row_count     50
91  2024-01-19 22:57:00+00:00   carts    log.row_count    100
90  2024-01-19 22:57:00+00:00   carts  log.error_count      0
126 2024-01-19 22:59:00+00:00   carts  log.error_count      0
127 2024-01-19 22:59:00+00:00   carts    log.row_count     25
146 2024-01-19 23:00:00+00:00   carts  log.error_count      0
147 2024-01-19 23:00:00+00:00   carts    log.row_count     75
164 2024-01-19 23:01:00+00:00   carts  log.error_count      0
165 2024-01-19 23:01:00+00:00   carts    log.row_count     25
182 2024-01-19 23:02:00+00:00   carts  log.error_count      0
183 2024-01-19 23:02:00+00:00   carts    log.row_count    125
200 2024-01-19 23:03:00+00:00   carts  log.error_count      0
201 2024-01-19 23:03:00+00:00   carts    log.row_count     50)```
```